In [3]:
# 1. Importer les bibliothèques nécessaires
%pip install pymongo pandas tqdm

from pymongo import MongoClient
import pandas as pd
import os

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   --------------------- ------------------ 524.3/972.8 kB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 972.8/972.8 kB 2.6 MB/s  0:00:00

   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   -------------------

In [5]:
# 2. Connexion à MongoDB


# On se connecte à la base créée dans 1_create.ipynb
client = MongoClient("mongodb://localhost:27017/")
db = client["Tweet_Worldcup"]
collection = db["tweets_raw"]

print("Nombre total de tweets :", collection.estimated_document_count())

Nombre total de tweets : 4569999


In [6]:
# 3. Extraire les variables utiles de chaque tweet par lots

import time

output_path = "../dataset/data_with_features.csv"
os.makedirs("../dataset", exist_ok=True)

start_time = time.time()

batch = []
batch_size = 100000
total = 0
first_write = True

cursor = collection.find({}, no_cursor_timeout=True).batch_size(1000)

for tweet in cursor:
    try:
        user = tweet["user"]
        entities = tweet["entities"]

        features = {
            # Variables utilisateur
            "user_id": user["id"],
            "screen_name": user["screen_name"],
            "followers_count": user["followers_count"],
            "friends_count": user["friends_count"],
            "statuses_count": user["statuses_count"],
            "verified": user["verified"],
            "default_profile": user["default_profile"],
            "default_profile_image": user["default_profile_image"],
            "favourites_count": user["favourites_count"],
            "listed_count": user["listed_count"],

            # Variables tweet
            "retweet_count": tweet.get("retweet_count", 0),
            "favorite_count": tweet.get("favorite_count", 0),
            "reply_count": tweet.get("reply_count", 0),
            "quote_count": tweet.get("quote_count", 0),
            "lang": tweet.get("lang", "unknown"),

            # Variables contenu
            "hashtags_count": len(entities.get("hashtags", [])),
            "urls_count": len(entities.get("urls", [])),
            "mentions_count": len(entities.get("user_mentions", [])),

            # Longueur du tweet
            "tweet_length": len(tweet.get("text", "")),

            # Type de tweet
            "is_retweet": 1 if "retweeted_status" in tweet else 0,
            "is_quote": 1 if tweet.get("is_quote_status", False) else 0
        }

        batch.append(features)
        total += 1

        if len(batch) >= batch_size:
            df_batch = pd.DataFrame(batch)

            df_batch.to_csv(
                output_path,
                mode="w" if first_write else "a",
                index=False,
                header=first_write,
                encoding="utf-8"
            )

            first_write = False
            batch = []

            print(f"{total} tweets traités et sauvegardés")

    except Exception as e:
        print("Erreur lors du traitement d'un tweet :", e)

cursor.close()

# Sauvegarder le dernier lot
if batch:
    df_batch = pd.DataFrame(batch)

    df_batch.to_csv(
        output_path,
        mode="w" if first_write else "a",
        index=False,
        header=first_write,
        encoding="utf-8"
    )

end_time = time.time()

print("Extraction terminée.")
print("Nombre total de tweets traités :", total)
print("Temps total :", round(end_time - start_time, 2), "secondes")
print("Fichier sauvegardé :", output_path)

100000 tweets traités et sauvegardés
200000 tweets traités et sauvegardés
300000 tweets traités et sauvegardés
400000 tweets traités et sauvegardés
500000 tweets traités et sauvegardés
600000 tweets traités et sauvegardés
700000 tweets traités et sauvegardés
800000 tweets traités et sauvegardés
900000 tweets traités et sauvegardés
1000000 tweets traités et sauvegardés
1100000 tweets traités et sauvegardés
1200000 tweets traités et sauvegardés
1300000 tweets traités et sauvegardés
1400000 tweets traités et sauvegardés
1500000 tweets traités et sauvegardés
1600000 tweets traités et sauvegardés
1700000 tweets traités et sauvegardés
1800000 tweets traités et sauvegardés
1900000 tweets traités et sauvegardés
2000000 tweets traités et sauvegardés
2100000 tweets traités et sauvegardés
2200000 tweets traités et sauvegardés
2300000 tweets traités et sauvegardés
2400000 tweets traités et sauvegardés
2500000 tweets traités et sauvegardés
2600000 tweets traités et sauvegardés
2700000 tweets traité